In this notebook the model will be chosen and after defyining the threshold i will create full pipeline that works with the raw data

In [100]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix,
    precision_score, recall_score, f1_score, classification_report)

from sklearn.preprocessing import (OneHotEncoder, 
                                   StandardScaler, 
                                   OrdinalEncoder,
                                   PowerTransformer)
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin


from sklearn.base import clone
import lightgbm as lgb
from lightgbm import LGBMClassifier


from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    cross_val_predict
)

import pandas as pd
import numpy as np  
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib
import json

In [149]:
BASE_PATH= Path(os.getcwd()).parent

DATASET_PATH= BASE_PATH / 'dataset'

TRAIN_RAW_PATH= DATASET_PATH / 'split' / 'raw' / 'train_set.csv'
TEST_RAW_PATH= DATASET_PATH / 'split' / 'raw' / 'test_set.csv'
TRAIN_PREPRO_PATH= DATASET_PATH / 'split' /  'preprocessed' /'train_set.csv'
TEST_PREPRO_PATH= DATASET_PATH / 'split' /  'preprocessed' /'test_set.csv'


ARTIFACTS_PATH= BASE_PATH / 'artifacts'
FEATURES_PATH= ARTIFACTS_PATH / 'final_features.json'
MODEL_DATA_PATH=ARTIFACTS_PATH / 'model_data'
CUSTOM_MODEL_PATH=MODEL_DATA_PATH / 'models' / 'full_custom_final_model.joblib'
PRECUSTOM_MODEL_PATH=MODEL_DATA_PATH / 'models' / 'full_precustom_final_model.joblib'
THRESHOLD_PATH=MODEL_DATA_PATH/'threshold_recall.json'


In [86]:
train_df=pd.read_csv(TRAIN_PREPRO_PATH)
test_df=pd.read_csv(TEST_PREPRO_PATH)

In [92]:
with open(FEATURES_PATH, "r", encoding='utf-8') as f:
    meta=json.load(f)

In [93]:
target=meta['target']
features=meta['final_features']
random_state=meta['random_state']
PAY_N=meta['PAY_N']
PAY_AMT=meta['PAY_AMT']
BILL_AMT=meta['BILL_AMT']

In [94]:
def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    ord_cols=PAY_N
    cat_cols = [c for c in features if c not in num_cols and c not in ord_cols]
    ordinal_categories = [[-2, -1, 0, 1, 2, 3]] * len(ord_cols)
    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True))
    ])
    ord_pipe= Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ordenc", OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ])
    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("ord", ord_pipe, ord_cols),
            ("cat", cat_pipe, cat_cols)
        ],
        remainder="drop"
    )
    return preprocessor


def make_pipeline(model, X: pd.DataFrame) -> Pipeline:
    preprocessor = make_preprocessor(X)
    return Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])


def evaluate_models_cv(
    X: pd.DataFrame,
    y: pd.Series,
    models: dict,
    n_splits: int = 5,
    seed: int = 42,
    return_oof: bool = True
) -> dict:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    results = {}
    for name, model in models.items():
        pipe = make_pipeline(model, X)

        # cross_validate gives per-fold scores + timing
        cv_out = cross_validate(
            pipe, X, y,
            cv=cv,
            scoring="roc_auc",
            return_train_score=False,
            n_jobs=-1
        )
        fold_scores = cv_out["test_score"]
        mean_auc = float(np.mean(fold_scores))
        std_auc = float(np.std(fold_scores))

        oof_auc = None
        if return_oof:
            # For AUC we need scores (probabilities or decision function)
            # cross_val_predict supports method='predict_proba' or 'decision_function'.
            # We try predict_proba first; if not available, fall back to decision_function.
            try:
                oof_scores = cross_val_predict(
                    pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1
                )[:, 1]
            except Exception:
                oof_scores = cross_val_predict(
                    pipe, X, y, cv=cv, method="decision_function", n_jobs=-1
                )
            oof_auc = float(roc_auc_score(y, oof_scores))

        results[name] = {
            "roc_auc_mean": mean_auc,
            "roc_auc_std": std_auc,
            "fold_scores": fold_scores,
            "oof_roc_auc": oof_auc,
            "fit_time_mean": float(np.mean(cv_out["fit_time"])),
            "score_time_mean": float(np.mean(cv_out["score_time"]))
        }

    return results


def print_cv_results(results: dict):
    rows = []
    for name, r in results.items():
        rows.append({
            "model": name,
            "roc_auc_mean": r["roc_auc_mean"],
            "roc_auc_std": r["roc_auc_std"],
            "oof_roc_auc": r["oof_roc_auc"],
            "fit_time_mean": r["fit_time_mean"]
        })
    summary = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
    print(summary.to_string(index=False))

    print("\nPer-fold scores:")
    for name, r in results.items():
        print(f"\n{name}: {np.round(r['fold_scores'], 4)}")

In [95]:
from xgboost import XGBClassifier

X=train_df.drop(columns=target, inplace=False)
y=train_df[target]

In [96]:
#training of different ml models with their default parameters
models = {
    "RandomForest": RandomForestClassifier(random_state=random_state),
    "GradientBoosting": GradientBoostingClassifier(random_state=random_state),
    "GaussianNB": GaussianNB(),
    "SVC_RBF": SVC(kernel="rbf", C=1.0, gamma="scale", random_state=random_state),
    "XGBClassifier": XGBClassifier(random_state=random_state, eval_metric="auc")
}

results = evaluate_models_cv(X, y, models=models, n_splits=5, seed=random_state, return_oof=True)
print_cv_results(results)

           model  roc_auc_mean  roc_auc_std  oof_roc_auc  fit_time_mean
GradientBoosting      0.780703     0.005989     0.780548       5.150905
   XGBClassifier      0.760814     0.005196     0.760807       0.612220
    RandomForest      0.760042     0.003088     0.760006       3.207353
      GaussianNB      0.738074     0.011513     0.738015       0.065685
         SVC_RBF      0.725136     0.008432     0.724943       8.638608

Per-fold scores:

RandomForest: [0.7595 0.7635 0.7561 0.7574 0.7637]

GradientBoosting: [0.784  0.7823 0.7688 0.7838 0.7846]

GaussianNB: [0.7536 0.7406 0.7191 0.7334 0.7437]

SVC_RBF: [0.7263 0.7267 0.7146 0.7393 0.7188]

XGBClassifier: [0.7634 0.7631 0.7514 0.7666 0.7596]


In [97]:
os.makedirs("catboost_tmp", exist_ok=True)

models={
    "LGBMClassifier": LGBMClassifier(random_state=random_state, verbose=-1),
}
results = evaluate_models_cv(X, y, models=models, n_splits=5, seed=random_state, return_oof=True)
print_cv_results(results)

         model  roc_auc_mean  roc_auc_std  oof_roc_auc  fit_time_mean
LGBMClassifier      0.779279     0.005427     0.779195       2.318614

Per-fold scores:

LGBMClassifier: [0.7814 0.7813 0.7685 0.7822 0.7831]


The best result can be seen from the GradientBoosting but still i am gonna choose the LGBMClassifier since it has much more tuning stuff that is more likely to increase the result

In [99]:
def sanity_check_shuffle_y(
    X: pd.DataFrame,
    y: pd.Series,
    model,
    seed: int = 42
) -> float:
    """
    Shuffle target; AUC should drop to ~0.50. If not, suspect leakage/bug.
    """
    rng = np.random.default_rng(seed)
    y_shuffled = pd.Series(rng.permutation(y.values), index=y.index)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    pipe = make_pipeline(model, X)

    scores = cross_validate(pipe, X, y_shuffled, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"]
    return float(np.mean(scores))

# Optional leakage sanity check on your best model:
leak_auc = sanity_check_shuffle_y(X, y, model=models["LGBMClassifier"], seed=42)
print("Shuffle-y sanity AUC (should be ~0.50):", leak_auc)

Shuffle-y sanity AUC (should be ~0.50): 0.4966397512514509


In [106]:
def make_objective(X: pd.DataFrame, y: pd.Series, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    def objective(trial: optuna.Trial) -> float:
        params = {
            "objective": "binary",
            "random_state": seed,
            "verbosity": -1,

            # фиксируем много деревьев и используем early stopping
            "n_estimators": 10000,

            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        }

        oof = np.zeros(len(y), dtype=float)

        for tr_idx, va_idx in cv.split(X, y):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            # Важно: строим pipeline на train-фолде (если у тебя dtype-логика внутри make_pipeline)
            model = LGBMClassifier(**params)
            pipe = make_pipeline(model, X_tr)  # твой pipeline (FE -> preprocess -> model)

            # 1) отдельно обучаем ВСЁ до модели и трансформим
            preproc = pipe[:-1]          # всё кроме последнего шага (модели)
            clf = pipe[-1]               # последняя модель (LGBMClassifier)

            X_tr_t = preproc.fit_transform(X_tr, y_tr)
            X_va_t = preproc.transform(X_va)

            # 2) обучаем модель на матрицах, и валидируем тоже на матрицах
            clf.fit(
                X_tr_t, y_tr,
                eval_set=[(X_va_t, y_va)],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(200, verbose=False)]
            )

            oof[va_idx] = clf.predict_proba(X_va_t)[:, 1]

        return roc_auc_score(y, oof)

    return objective

In [113]:
import warnings
messages=[
    "No further splits with positive gain, best gain: -inf",
    "X does not have valid feature names, but LGBMClassifier was fitted with feature names"
]
for msg in messages:
    warnings.filterwarnings(
        "ignore",
    message=msg
    )

In [109]:
# ---------- Run Optuna ----------
def tune_xgb_optuna(X, y, n_trials=50, seed=random_state):
    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(make_objective(X, y, n_splits=5, seed=seed), n_trials=n_trials)

    print("Best AUC:", study.best_value)
    print("Best params:", study.best_params)
    return study

X = train_df.drop(columns=[target])
y = train_df[target]

study = tune_xgb_optuna(X, y, n_trials=50, seed=42)

[I 2026-02-08 18:42:25,095] A new study created in memory with name: no-name-7bd01c2c-1c17-4757-81dc-13b174fe073d
[I 2026-02-08 18:42:31,586] Trial 0 finished with value: 0.7775368483157888 and parameters: {'learning_rate': 0.02757359293934948, 'num_leaves': 230, 'min_child_samples': 152, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'reg_lambda': 0.000602521573620386}. Best is trial 0 with value: 0.7775368483157888.
[I 2026-02-08 18:42:43,621] Trial 1 finished with value: 0.7847559277604907 and parameters: {'learning_rate': 0.011703388679635262, 'num_leaves': 192, 'min_child_samples': 128, 'subsample': 0.9124217733388136, 'colsample_bytree': 0.7061753482887407, 'reg_lambda': 7.072114131472227}. Best is trial 1 with value: 0.7847559277604907.
[I 2026-02-08 18:42:48,310] Trial 2 finished with value: 0.7834414430503986 and parameters: {'learning_rate': 0.09528587217040241, 'num_leaves': 48, 'min_child_samples': 52, 'subsample': 0.7550213529560301, 'colsample_by

Best AUC: 0.7864658250956039
Best params: {'learning_rate': 0.010060697427449644, 'num_leaves': 43, 'min_child_samples': 121, 'subsample': 0.7267409340764938, 'colsample_bytree': 0.770882382469551, 'reg_lambda': 9.01339323745404}


In [110]:
best_params=study.best_params

In [111]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

best_model = XGBClassifier(
    **study.best_params,
    eval_metric="logloss",
    tree_method="hist",
    random_state=random_state,
    n_jobs=-1
)

pipe = make_pipeline(best_model, X)  # uses your make_pipeline + preprocessing

oof_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

print("OOF ROC AUC:", roc_auc_score(y, oof_proba))
print("OOF PR AUC (Average Precision):", average_precision_score(y, oof_proba))

OOF ROC AUC: 0.7797280895003684
OOF PR AUC (Average Precision): 0.549852898936798


In [112]:
#saving the best hyperparameters
with open(MODEL_DATA_PATH / "best_params.json", "w", encoding="utf-8") as f:
    json.dump(study.best_params, f, indent=4)

In [114]:
def train_final_once(df: pd.DataFrame, target: str, best_params: dict, seed: int = 42):
    X = df.drop(columns=[target])
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    final_params = dict(best_params)
    final_params.update({
        "random_state": seed,
        "n_jobs": -1
    })

    model = LGBMClassifier(**final_params)
    pipe = make_pipeline(model, X_train)
    pipe.fit(X_train, y_train)

    test_scores = pipe.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_scores)

    print("FINAL TEST AUC:", test_auc)
    return pipe, test_auc

final_pipe, test_auc = train_final_once(train_df, target, best_params=best_params, seed=random_state)

FINAL TEST AUC: 0.7801339679315303


In [115]:
def pick_threshold_for_recall(y_true, proba_pos, target_recall=0.85):
    prec, rec, thr = precision_recall_curve(y_true, proba_pos)

    # thr length = len(prec)-1, so align:
    prec2, rec2 = prec[1:], rec[1:]

    mask = rec2 >= target_recall
    if not mask.any():
        # если заданный recall недостижим, берём максимум recall
        idx = np.argmax(rec2)
        t = thr[idx]
        return t, prec2[idx], rec2[idx]

    # среди порогов, где recall >= target, берём максимум precision
    idx = np.argmax(prec2[mask])
    t = thr[mask][idx]
    p = prec2[mask][idx]
    r = rec2[mask][idx]
    return t, p, r


# пример использования:
target_recall = 0.85
t, p, r = pick_threshold_for_recall(y, oof_proba, target_recall=target_recall)
print("Chosen threshold:", t)
print("Precision:", p)
print("Recall:", r)

Chosen threshold: 0.16158445
Precision: 0.3137295794230101
Recall: 0.8500659257864005


Chosing the threshold as recall and computing using the oof. I want the threshold to be around the 0.85

Note: since we are trying to balance and also maximize the recall the other metrics will drop such as precision for class 1 will drop

In [116]:
y_pred = (oof_proba >= t).astype(int)
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

print("TN FP FN TP:", tn, fp, fn, tp)
print("F1:", f1_score(y, y_pred))
print("Flag rate:", y_pred.mean())

TN FP FN TP: 8818 9873 796 4513
F1: 0.4582889058136583
Flag rate: 0.5994166666666667


In [117]:
threshold_recall=t

In [118]:
pred = (oof_proba >= threshold_recall).astype(int)

print("Confusion matrix:\n", confusion_matrix(y, pred))
print("Precision:", precision_score(y, pred))
print("Recall:", recall_score(y, pred))
print("F1:", f1_score(y, pred))
print("\nClassification report:\n", classification_report(y, pred))

Confusion matrix:
 [[8818 9873]
 [ 796 4513]]
Precision: 0.3137077714444599
Recall: 0.8500659257864005
F1: 0.4582889058136583

Classification report:
               precision    recall  f1-score   support

           0       0.92      0.47      0.62     18691
           1       0.31      0.85      0.46      5309

    accuracy                           0.56     24000
   macro avg       0.62      0.66      0.54     24000
weighted avg       0.78      0.56      0.59     24000



In [119]:
threshold={
    'threshold_recall': float(threshold_recall)
}

In [120]:
with open(THRESHOLD_PATH, "w", encoding='utf-8') as f:
    json.dump(threshold, f, indent=2)

In [121]:
class DeafaultFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, bill_cols, method="yeo-johnson", standardize=False):
        self.bill_cols = list(bill_cols)
        self.method = method
        self.standardize = standardize
    def fit(self, X: pd.DataFrame, y=None):
        X = X.copy()
        self.pt_ = PowerTransformer(method=self.method, standardize=self.standardize)
        self.pt_.fit(X[self.bill_cols])
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        for col in PAY_N:
            X[col] = X[col].clip(upper=3)
        X["LIMIT_BAL_LOG"]=np.log1p(X["LIMIT_BAL"])
        X["EDUCATION"]=X["EDUCATION"].replace({1:'graduate school', 2:'university', 
                                               3:'high school', 4:'others', 
                                               5:'others', 6:'others', 0:'others'})
        X['SEX']=X['SEX'].replace({1:'male', 2:'female'})
        X['MARRIAGE']=X['MARRIAGE'].replace({1:'married', 2:'single', 3:'others', 0:'others'})
        bins   = [21, 25, 30, 35, 45, np.inf]
        labels = ["21-25", "25-30", "30-35", "35-45", "45+"]
        X["AGE_BIN"] = pd.cut(X["AGE"], bins=bins, labels=labels)
        for col in PAY_AMT:
            X[col] = np.log1p(X[col])
        X[self.bill_cols]=self.pt_.transform(X[self.bill_cols])
        return X


def build_custom_pipeline(best_params: dict, random_state: int = 42, final_pipe=None) -> Pipeline:
    
   

    return Pipeline([
        ("fe", DeafaultFeatureEngineer(bill_cols=BILL_AMT, method="yeo-johnson", standardize=False)),
        ("pipe", final_pipe)
    ])

here we have the full ready pipeline with custom featureengineering. to check if the performance is the same with and without the custom featureengineering i would compare both of them where to one i will give the raw and for one the proccessed. 
>Note: raw and processed must be the same data

In [137]:
def check_similarity_finalpipe(pipe, train_df, target):
    X = train_df.drop(columns=[target])
    y = train_df[target]

    proba = pipe.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    print("Test AUC:", auc)

In [138]:
def check_similarity_custom_finalpipe(pipe, train_df, target):
    X = train_df.drop(columns=[target])
    y = train_df[target]

    pipe.fit(X, y)
    
    proba = pipe.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, proba)
    print("Test AUC:", auc)

In [139]:
check_similarity_finalpipe(final_pipe, train_df, target)

Test AUC: 0.7902754847024431


In [140]:
train_raw_df=pd.read_csv(TRAIN_RAW_PATH)
custom_pipeline=build_custom_pipeline(best_params=best_params, random_state=random_state, final_pipe=final_pipe)
check_similarity_custom_finalpipe(custom_pipeline, train_raw_df, target)

Test AUC: 0.8083315980640997


The results should not be the same but similar. I mean like even if we just added the custom feature engineering class for the final_pipe the result may change due to other order of giving the data or etc.

result on test datasets

In [141]:
check_similarity_finalpipe(final_pipe, test_df, target)

Test AUC: 0.7720955783283243


In [147]:
test_raw_df=pd.read_csv(TEST_RAW_PATH)
proba=custom_pipeline.predict_proba(test_raw_df.drop(columns=[target]))[:, 1]
metric_on_test=roc_auc_score(test_raw_df[target], proba)
print("Final AUC on test set:", metric_on_test)

Final AUC on test set: 0.7755540131696605


Quite similar

In [150]:
os.makedirs(CUSTOM_MODEL_PATH.parent, exist_ok=True)
joblib.dump(custom_pipeline, CUSTOM_MODEL_PATH)

['c:\\Users\\User\\all_project\\projects_in_github\\taiwan2005_credict_card_project\\artifacts\\model_data\\models\\full_custom_final_model.joblib']

In [152]:
os.makedirs(PRECUSTOM_MODEL_PATH.parent, exist_ok=True)
joblib.dump(final_pipe, PRECUSTOM_MODEL_PATH)

['c:\\Users\\User\\all_project\\projects_in_github\\taiwan2005_credict_card_project\\artifacts\\model_data\\models\\full_precustom_final_model.joblib']